In [ ]:
import os
import warnings

import duckdb
import pandas as pd
from haversine import haversine_vector, Unit

warnings.filterwarnings("ignore")

In [ ]:
#path = r"C:\Users\bhavy\Massachusetts Institute of Technology\Truck Parking Capstone - General\Truck Stop Finder 🚚⛽\\"
path = r"C:\Users\samcl\Massachusetts Institute of Technology\Truck Parking Capstone - Truck Stop Finder 🚚⛽\\"

In [ ]:
sensor_loc = pd.read_csv(path + r"5. Source & Refrence Files\sensor_loc.csv",
                         dtype={"station_id": 'str'})
state_map = pd.read_csv(path + r"5. Source & Refrence Files\State_mapping.csv")
model_stop = pd.read_excel("output_excel/Model_Stops_V3.xlsx")
traffic_df = pd.read_csv(path + r"4. Working Data Files\Traffic Files\Capstone_truck\merged_filtered_modified.csv")


In [ ]:
sensor_loc.columns

In [ ]:
traffic_df["link_id"] = traffic_df["routeid"].astype("str") + "_" + traffic_df["beginpoint"].astype("str") + "_" + \
                        traffic_df["endpoint"].astype("str")
in_link = traffic_df[traffic_df["link_id"].isin(model_stop["link_id"].unique())][["link_id", "MID_LAT", "MID_LONG"]]

sensor_stops = pd.merge(in_link, sensor_loc, how="cross")

sensor_stops['distance_miles'] = haversine_vector(sensor_stops[['Latitude', 'Longitude']],
                                                  sensor_stops[['MID_LAT', 'MID_LONG']],
                                                  Unit.MILES
                                                  ) * 1.16

#Finding nearest road segment to every stop
idx = sensor_stops.groupby('link_id')['distance_miles'].idxmin()
nearest = sensor_stops.loc[idx].reset_index()
nearest = nearest[['Latitude', 'Longitude', 'Functional Class', 'State', 'Station Id', 'link_id']].copy()
nearest

In [ ]:
# folder = path + r"5. Source & Refrence Files\2024_traffic_data"
# out_dir = os.path.join(path, r"5. Source & Refrence Files\2024_traffic_parquet")
# os.makedirs(out_dir, exist_ok=True)
#
# # build mapping dict ONCE from your state_map df
# state_dict = state_map.set_index("state_code")["State"].to_dict()
#
# id_var_col = [
#     'record_type', 'state_code', 'f_system', 'station_id', 'travel_dir',
#     'travel_lane', 'year_record', 'month_record', 'day_record',
#     'day_of_week', 'restrictions'
# ]
#
# part = 0
#
# for filename in os.listdir(folder):
#     file_path = os.path.join(folder, filename)
#
#     if not filename.lower().endswith(".zip"):
#         continue
#
#     print(f"Opening ZIP: {filename}")
#
#     with zipfile.ZipFile(file_path, 'r') as z:
#         for inner_name in z.namelist():
#             if inner_name.endswith("/"):
#                 continue
#
#             print(f"  Processing inside ZIP: {inner_name}")
#
#             with z.open(inner_name) as f:
#                 # Read CSV from inside the ZIP
#                 df = pd.read_csv(
#                     f,
#                     delimiter="|",
#                     low_memory=False  # avoids dtype warning at cost of some RAM, OK for chunk
#                     # You can also pass dtype={} here if you know them
#                 )
#
#                 # Melt wide hours columns into long format
#                 df = pd.melt(
#                     df,
#                     id_vars=id_var_col,
#                     var_name="hours",
#                     value_name="traffic"
#                 )
#
#                 # Add state_name via map instead of big merge later
#                 df["State"] = df["state_code"].map(state_dict)
#                 df["station_id"] = df["station_id"].astype(str)
#
#                 # Save this chunk to Parquet and drop from memory
#                 out_path = os.path.join(out_dir, f"traffic_part_{part}.parquet")
#                 df.to_parquet(out_path, index=False)
#                 # print(f"    → wrote {out_path}")
#
#                 del df
#                 part += 1
#
# print(f"Done. Wrote {part} parquet files to {out_dir}")


In [ ]:
traffic_df = pd.DataFrame()

In [ ]:
out_dir = os.path.join(path, r"5. Source & Refrence Files\2024_traffic_parquet")
traffic_parquet_glob = f"{out_dir}/traffic_part_*.parquet"

In [ ]:
con = duckdb.connect()
con.register("sensor_loc", nearest)

In [ ]:

out_dir = os.path.join(path, r"5. Source & Refrence Files\2024_traffic_parquet")
traffic_parquet_glob = f"{out_dir}/traffic_part_*.parquet"

In [ ]:
sensor_loc

In [ ]:
con.execute(f"""
    CREATE OR REPLACE TABLE traffic_matched AS
    SELECT
        t.*,                           -- all columns from traffic
        s.*
    FROM read_parquet('{traffic_parquet_glob}') AS t
    LEFT JOIN sensor_loc AS s
      ON t.station_id = s."Station Id"
     AND t.State      = s.State
    WHERE s."Latitude" IS NOT NULL
""")


In [ ]:
con.execute(f"""
    CREATE OR REPLACE TABLE traffic_unmatched  AS
    SELECT
        t.*,                           -- all columns from traffic
        s.*
    FROM read_parquet('{traffic_parquet_glob}') AS t
    LEFT JOIN sensor_loc AS s
      ON t.station_id = s."Station Id"
     AND t.State      = s.State
    WHERE s."Latitude" IS NULL
""")


In [ ]:
con.execute("""
            UPDATE traffic_unmatched
            SET station_id = ltrim(station_id, '0')
            """)


In [ ]:
con.execute("""select *
               from sensor_loc limit 5""").df()

In [ ]:
con.execute('ALTER TABLE traffic_unmatched DROP COLUMN Latitude')
con.execute('ALTER TABLE traffic_unmatched DROP COLUMN Longitude')
con.execute('ALTER TABLE traffic_unmatched DROP COLUMN "Functional Class"')
con.execute('ALTER TABLE traffic_unmatched DROP COLUMN State_1')
con.execute('ALTER TABLE traffic_unmatched DROP COLUMN "Station Id"')


In [ ]:
con.execute('ALTER TABLE traffic_unmatched DROP COLUMN link_id')

In [ ]:
con.execute("""select *
               from traffic_unmatched limit 5""").df()

In [ ]:
con.execute("""
            INSERT INTO traffic_matched
            SELECT t.*,
                   s.*
            FROM traffic_unmatched t
                     LEFT JOIN sensor_loc s
                               ON t.station_id = s."Station Id"
                                   AND t.State = s.State
            WHERE s."Latitude" IS NOT NULL
            """)


In [ ]:
con.execute("""
            CREATE OR REPLACE TABLE traffic_unmatched  AS
            SELECT t.*,
                   s.*
            FROM traffic_unmatched t
                     LEFT JOIN sensor_loc s
                               ON t.station_id = s."Station Id"
                                   AND t.State = s.State
            WHERE s."Latitude" IS NULL
            """)

In [ ]:
con.execute("""select count(distinct station_id)
               from traffic_unmatched """).df()

In [ ]:
con.execute("""select *
               from traffic_matched LIMIT 5""").df()

In [ ]:
GA = con.execute("""select *
                    from traffic_matched
                    where station_id = '00R348'""").df()



In [ ]:
GA.to_csv('GA_Sensor_Data.csv', index=False)

In [ ]:
con.execute("""select distinct travel_lane
               from traffic_matched
            """).df()

In [ ]:
con.execute("""select *
               from traffic_matched
               where travel_lane = 7""").df()

In [ ]:
con.execute("""select *
               from traffic_matched
               where station_id = '860331'""").df().to_csv('FL_example.csv', index=False)

In [ ]:
con.execute(""" select station_id, state_code, count(distinct travel_dir)
                from traffic_matched
                group by station_id, state_code
                having count(distinct travel_dir) > 1""").df()

In [ ]:
con.execute("""select *
               from traffic_matched
               where station_id = '186'""").df().to_csv('186.csv', index=False)